# Beginner 03 — Authentication, Credentials and Tokens

## Scenario

A procurement agent needs to authenticate to an internal purchasing API.

We start with API keys, then move to asymmetric keys, JWT validation, JWKS rotation, X.509, replay, and proof-of-possession.

> This notebook is educational. Production identity protocols should use mature standards and libraries rather than custom token formats.


In [ ]:
from datetime import datetime, timedelta, timezone
import base64, hashlib, json, secrets, uuid

import jwt
from cryptography import x509
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.x509.oid import NameOID

def now():
    return datetime.now(timezone.utc)

print("PyJWT:", jwt.__version__)


## 1 — API keys: simple bearer secrets

In [ ]:
API_KEYS = {
    "demo-key-agent-procurement": {
        "principal": "agent:procurement",
        "scopes": {"purchase:read"}
    }
}

def authenticate_api_key(key):
    record = API_KEYS.get(key)
    if not record:
        raise PermissionError("invalid API key")
    return record

print(authenticate_api_key("demo-key-agent-procurement"))


The key authenticates by possession. If copied, the copy works. It has no inherent expiry, audience, issuer, or proof-of-possession semantics unless the surrounding system adds them.

## 2 — Generate an asymmetric RSA key pair

In [ ]:
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

private_pem = private_key.private_bytes(
    serialization.Encoding.PEM,
    serialization.PrivateFormat.PKCS8,
    serialization.NoEncryption(),
)
public_pem = public_key.public_bytes(
    serialization.Encoding.PEM,
    serialization.PublicFormat.SubjectPublicKeyInfo,
)

print(public_pem.decode()[:180] + "...")


## 3 — Issue a short-lived signed JWT

In [ ]:
ISSUER = "https://identity.corp.example"
AUDIENCE = "purchasing-api"

issued = now()
claims = {
    "iss": ISSUER,
    "sub": "agent:procurement",
    "aud": AUDIENCE,
    "iat": issued,
    "nbf": issued,
    "exp": issued + timedelta(minutes=5),
    "jti": str(uuid.uuid4()),
    "scope": "purchase:read purchase:create",
}

token = jwt.encode(
    claims,
    private_pem,
    algorithm="RS256",
    headers={"kid": "key-2026-08", "typ": "JWT"},
)
print(token)


## 4 — Decode without verification: useful for debugging, unsafe for authentication

In [ ]:
unverified = jwt.decode(token, options={"verify_signature": False})
print(json.dumps(unverified, indent=2, default=str))


Anyone can construct a payload. Never turn unverified claims into an authenticated principal.

## 5 — Correct validation

In [ ]:
verified = jwt.decode(
    token,
    public_pem,
    algorithms=["RS256"],
    issuer=ISSUER,
    audience=AUDIENCE,
    options={"require": ["exp", "iss", "sub", "aud"]},
)
print("Authenticated subject:", verified["sub"])


## 6 — Wrong audience must fail

In [ ]:
try:
    jwt.decode(
        token, public_pem,
        algorithms=["RS256"],
        issuer=ISSUER,
        audience="payroll-api",
    )
except Exception as e:
    print(type(e).__name__, "->", e)


## 7 — Tampering must fail

In [ ]:
parts = token.split(".")
payload = json.loads(base64.urlsafe_b64decode(parts[1] + "=="))
payload["sub"] = "agent:admin"
new_payload = base64.urlsafe_b64encode(
    json.dumps(payload, separators=(",", ":")).encode()
).decode().rstrip("=")
tampered = ".".join([parts[0], new_payload, parts[2]])

try:
    jwt.decode(tampered, public_pem, algorithms=["RS256"], audience=AUDIENCE, issuer=ISSUER)
except Exception as e:
    print(type(e).__name__, "-> tampering detected")


## 8 — Expired credentials must fail

In [ ]:
expired = jwt.encode(
    {
        "iss": ISSUER, "sub": "agent:procurement", "aud": AUDIENCE,
        "iat": now() - timedelta(minutes=10),
        "exp": now() - timedelta(minutes=5),
    },
    private_pem, algorithm="RS256",
)
try:
    jwt.decode(expired, public_pem, algorithms=["RS256"], issuer=ISSUER, audience=AUDIENCE)
except Exception as e:
    print(type(e).__name__, "->", e)


## 9 — Algorithm allowlisting

Notice that every verification call supplies `algorithms=["RS256"]`.

The trusted algorithm policy comes from our application configuration—not from the token header.


## 10 — Build a public JWK

In [ ]:
def b64u_int(value: int) -> str:
    length = (value.bit_length() + 7) // 8
    return base64.urlsafe_b64encode(value.to_bytes(length, "big")).decode().rstrip("=")

numbers = public_key.public_numbers()
jwk = {
    "kty": "RSA",
    "kid": "key-2026-08",
    "use": "sig",
    "alg": "RS256",
    "n": b64u_int(numbers.n),
    "e": b64u_int(numbers.e),
}
jwks = {"keys": [jwk]}
print(json.dumps({**jwk, "n": jwk["n"][:40] + "..."}, indent=2))


## 11 — Simulate signing-key rotation

In [ ]:
old_private = private_key
old_public = public_key
new_private = rsa.generate_private_key(public_exponent=65537, key_size=2048)
new_public = new_private.public_key()

KEYS = {
    "key-A": old_public,
    "key-B": new_public,
}

rotated_token = jwt.encode(
    {
        "iss": ISSUER, "sub": "agent:procurement", "aud": AUDIENCE,
        "iat": now(), "exp": now() + timedelta(minutes=5),
    },
    new_private,
    algorithm="RS256",
    headers={"kid": "key-B"},
)

header = jwt.get_unverified_header(rotated_token)
selected_key = KEYS[header["kid"]]
print("Selected:", header["kid"])
print(jwt.decode(rotated_token, selected_key, algorithms=["RS256"], issuer=ISSUER, audience=AUDIENCE)["sub"])


During rotation, verifiers may need both old and new public keys until old tokens expire. `kid` selects among already-trusted issuer keys; it must not turn arbitrary attacker key material into trust.

## 12 — Create a self-signed X.509 certificate for learning

In [ ]:
from datetime import datetime as dt

cert_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, "agent:procurement")])

cert = (
    x509.CertificateBuilder()
    .subject_name(name)
    .issuer_name(name)
    .public_key(cert_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(dt.now(timezone.utc) - timedelta(minutes=1))
    .not_valid_after(dt.now(timezone.utc) + timedelta(hours=1))
    .add_extension(
        x509.SubjectAlternativeName([
            x509.UniformResourceIdentifier("spiffe://corp.example/prod/procurement")
        ]),
        critical=False,
    )
    .sign(cert_key, hashes.SHA256())
)

print("Subject:", cert.subject.rfc4514_string())
print("Issuer :", cert.issuer.rfc4514_string())
print("SAN    :", cert.extensions.get_extension_for_class(x509.SubjectAlternativeName).value)


This self-signed certificate is **not production trust**. Production validation needs a trusted CA/bundle, correct identity/SAN rules, validity and usage checks, and proof of private-key possession. The exercise exists to expose certificate structure.

## 13 — Bearer replay demonstration

In [ ]:
BEARER = secrets.token_urlsafe(24)

def bearer_api(presented):
    return "ALLOW" if secrets.compare_digest(presented, BEARER) else "DENY"

print("Original holder:", bearer_api(BEARER))
stolen_copy = BEARER
print("Attacker copy  :", bearer_api(stolen_copy))


The API cannot distinguish the original holder from a thief who possesses the same bearer secret.

## 14 — Simplified proof-of-possession concept

In [ ]:
pop_private = rsa.generate_private_key(public_exponent=65537, key_size=2048)
pop_public = pop_private.public_key()

challenge = secrets.token_bytes(32)
signature = pop_private.sign(
    challenge,
    padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
    hashes.SHA256(),
)

pop_public.verify(
    signature,
    challenge,
    padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
    hashes.SHA256(),
)
print("Proof verified: caller demonstrated possession of private key")


Real DPoP is a standardized OAuth protocol and contains request-bound fields, replay controls, and key-binding semantics. This cell demonstrates only the cryptographic intuition.

## 15 — Hardened JWT validator

In [ ]:
def validate_agent_token(token, key, *, issuer, audience):
    return jwt.decode(
        token,
        key,
        algorithms=["RS256"],
        issuer=issuer,
        audience=audience,
        options={
            "require": ["iss", "sub", "aud", "exp", "iat"],
            "verify_signature": True,
            "verify_exp": True,
            "verify_nbf": True,
            "verify_iss": True,
            "verify_aud": True,
        },
    )

print(validate_agent_token(token, public_pem, issuer=ISSUER, audience=AUDIENCE)["sub"])


## 16 — Security regression tests

In [ ]:
tests = []

def expect_failure(name, fn):
    try:
        fn()
        tests.append((name, False))
    except Exception:
        tests.append((name, True))

expect_failure("wrong audience", lambda: validate_agent_token(
    token, public_pem, issuer=ISSUER, audience="payroll-api"
))
expect_failure("wrong issuer", lambda: validate_agent_token(
    token, public_pem, issuer="https://evil.example", audience=AUDIENCE
))
expect_failure("tampered token", lambda: validate_agent_token(
    tampered, public_pem, issuer=ISSUER, audience=AUDIENCE
))
expect_failure("expired token", lambda: validate_agent_token(
    expired, public_pem, issuer=ISSUER, audience=AUDIENCE
))

for name, passed in tests:
    print(f"{name:18} {'PASS' if passed else 'FAIL'}")

assert all(passed for _, passed in tests)


## 17 — Challenge: implement a key-set validator

Create a validator that:

1. reads `kid` from the unverified header;
2. rejects missing/unknown `kid`;
3. selects a public key only from a preconfigured trusted dictionary;
4. validates RS256;
5. validates issuer and audience;
6. requires `sub`, `exp`, `iat`, `iss`, `aud`;
7. rejects tokens longer than your maximum acceptable lifetime.

Then write negative tests for:
- unknown key ID;
- wrong audience;
- expired token;
- excessive lifetime;
- tampered payload.


In [ ]:
def validate_from_keyset(token, trusted_keys, *, issuer, audience, max_lifetime_seconds=600):
    # Implement as an exercise.
    raise NotImplementedError


## 18 — Design challenge: credential placement

For each item below decide whether the LLM should ever see the raw credential and explain why:

- API key for Salesforce;
- user's OAuth access token;
- SPIFFE X.509 private key;
- MCP server bearer token;
- public JWKS;
- certificate public chain;
- DPoP private key.

Then design:

```text
LLM -> Tool Gateway -> Credential Broker -> API
```

so the model can request an operation without receiving the credential.

## Review questions

1. What is the difference between identity and credential?
2. Why is decoding a JWT not authentication?
3. Why must audience be validated?
4. Why should algorithms be allowlisted?
5. Why is a signed JWT not confidential?
6. What problem does `kid` solve?
7. Why are old public keys temporarily retained during rotation?
8. What is the primary replay weakness of bearer credentials?
9. How does proof-of-possession change the attacker's requirements?
10. Why are short-lived workload credentials preferable to embedded API keys?

## Next course

**Beginner 04 — Authorization for Agents**
